# Predicting Student Performance using Machine Learning Algorithms with TensorFlow

## Project Overview
This notebook analyzes the Student Performance dataset to:
- Explore relationships between student attributes and academic results
- Build predictive models using TensorFlow/Keras
- Identify key factors influencing performance

---

## Step 1: Environment Setup and Library Imports

In [ ]:
# Data manipulation and analysis
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning - preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression

# Machine learning - evaluation metrics
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import roc_curve, auc, roc_auc_score

# Deep learning with TensorFlow
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline
pd.set_option('display.max_columns', None)
np.random.seed(42)
tf.random.set_seed(42)

print("✓ All libraries imported successfully!")

## Step 2: Load the Dataset

In [ ]:
# Load dataset
df = pd.read_csv(r'C:\Users\ABDULAI IDDRISU\Desktop\ultimate_student_productivity_dataset_5000.csv')
print("✓ Dataset loaded successfully!")
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## Step 3: Inspect Dataset Structure

In [ ]:
# Dataset information
print("Column names and types:")
print(df.dtypes)
print("\nBasic statistics:")
df.describe()

In [ ]:
# Check missing values
missing = df.isnull().sum()
if missing.sum() > 0:
    print("Missing values:")
    print(missing[missing > 0])
else:
    print("✓ No missing values!")

## Step 4: Data Cleaning and Preparation

In [ ]:
# Create clean copy
df_clean = df.copy()

# Identify column types
numerical_cols = df_clean.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = df_clean.select_dtypes(include=['object']).columns.tolist()

print(f"Numerical columns ({len(numerical_cols)}): {numerical_cols}")
print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")

In [ ]:
# Encode categorical variables
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))
    label_encoders[col] = le
    print(f"✓ Encoded '{col}'")

print("\n✓ All categorical variables encoded!")
df_clean.head()

## Step 5: Define Target Variable

**IMPORTANT**: You need to identify which column is your target (what you want to predict).
Common target names: 'Grade', 'Performance', 'Final_Grade', 'G3', 'Score', etc.

Update the `target_column` variable below with your actual target column name.

In [ ]:
# CHANGE THIS to your actual target column name
target_column = 'Grade'  # Update this!

# Check if target exists
if target_column in df_clean.columns:
    print(f"✓ Target column '{target_column}' found")
    print(f"\nTarget statistics:")
    print(f"Min: {df_clean[target_column].min()}")
    print(f"Max: {df_clean[target_column].max()}")
    print(f"Unique values: {df_clean[target_column].nunique()}")
    
    # If too many unique values, convert to categories
    if df_clean[target_column].nunique() > 10:
        print(f"\n⚠ Too many unique values ({df_clean[target_column].nunique()}). Converting to performance categories...")
        
        # Create performance categories based on quartiles
        df_clean[target_column + '_Category'] = pd.qcut(df_clean[target_column], 
                                                        q=4, 
                                                        labels=['Low', 'Medium', 'High', 'Excellent'],
                                                        duplicates='drop')
        
        # Use the categorical version as target
        target_column = target_column + '_Category'
        print(f"\n✓ Created categorical target: {target_column}")
    
    print(f"\nFinal target distribution:")
    print(df_clean[target_column].value_counts().sort_index())
    
else:
    print(f"❌ Column '{target_column}' not found!")
    print(f"\nAvailable columns: {df_clean.columns.tolist()}")

## Step 6: Exploratory Data Analysis (EDA)

In [ ]:
# Target distribution visualization
plt.figure(figsize=(10, 6))
target_counts = df_clean[target_column].value_counts().sort_index()
target_counts.plot(kind='bar', color='skyblue')
plt.title(f'Distribution of {target_column}', fontsize=14, fontweight='bold')
plt.xlabel(target_column)
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"\nTarget distribution:")
print(target_counts)

In [ ]:
# Correlation heatmap (top 15 features)
plt.figure(figsize=(12, 10))
corr_matrix = df_clean.corr()
top_corr = corr_matrix[target_column].abs().sort_values(ascending=False)[:15]
top_features = top_corr.index.tolist()
sns.heatmap(df_clean[top_features].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap - Top 15 Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nTop 10 features correlated with target:")
print(top_corr[1:11])

## Step 7: Feature Selection and Train-Test Split

In [ ]:
# Separate features and target
X = df_clean.drop(columns=[target_column])
y = df_clean[target_column]

print(f"Original Features shape: {X.shape}")
print(f"Original Target shape: {y.shape}")
print(f"\nAll target classes: {sorted(y.unique())}")
print(f"Number of classes: {len(y.unique())}")

In [ ]:
# Check class distribution first
class_counts = y.value_counts()
print("Class distribution:")
print(class_counts)

# Remove classes with too few samples (less than 2)
valid_classes = class_counts[class_counts >= 2].index
mask = y.isin(valid_classes)
X_filtered = X[mask]
y_filtered = y[mask]

print(f"\nOriginal samples: {len(y)}")
print(f"Filtered samples: {len(y_filtered)}")
print(f"Classes with >=2 samples: {len(valid_classes)}")

# Split data: 80% training, 20% testing
# Use stratify only if all classes have at least 2 samples
if (y_filtered.value_counts() >= 2).all():
    X_train, X_test, y_train, y_test = train_test_split(X_filtered, y_filtered, test_size=0.2, random_state=42, stratify=y_filtered)
    print("\n✓ Used stratified split")
else:
    X_train, X_test, y_train, y_test = train_test_split(X_filtered, y_filtered, test_size=0.2, random_state=42)
    print("\n✓ Used random split (no stratification)")

print(f"\nTraining set: {X_train.shape[0]} samples")
print(f"Testing set: {X_test.shape[0]} samples")
print(f"\nTraining target distribution:")
print(y_train.value_counts())

## Step 8: Feature Scaling

In [ ]:
# Standardize features (mean=0, std=1)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✓ Features scaled successfully!")
print(f"\nScaled training data shape: {X_train_scaled.shape}")
print(f"Mean: {X_train_scaled.mean():.4f}")
print(f"Std: {X_train_scaled.std():.4f}")

## Step 9: Baseline Model - Logistic Regression

In [ ]:
# Train Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)

# Predictions
y_pred_lr = lr_model.predict(X_test_scaled)

# Evaluate
lr_accuracy = accuracy_score(y_test, y_pred_lr)
print("=" * 60)
print("LOGISTIC REGRESSION RESULTS")
print("=" * 60)
print(f"\nAccuracy: {lr_accuracy:.4f} ({lr_accuracy*100:.2f}%)")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr))

In [ ]:
# Confusion Matrix
cm_lr = confusion_matrix(y_test, y_pred_lr)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues')
plt.title('Logistic Regression - Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

## Step 10: Build TensorFlow Neural Network

In [ ]:
# Determine number of classes
num_classes = len(y.unique())
input_dim = X_train_scaled.shape[1]

print(f"Input features: {input_dim}")
print(f"Output classes: {num_classes}")

# Build neural network
model = Sequential([
    Dense(128, activation='relu', input_dim=input_dim),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(num_classes, activation='softmax' if num_classes > 2 else 'sigmoid')
])

# Compile model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy' if num_classes > 2 else 'binary_crossentropy',
    metrics=['accuracy']
)

print("\n✓ Neural network built successfully!")
model.summary()

## Step 11: Train Neural Network

In [ ]:
# Early stopping to prevent overfitting
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Train model
print("Training neural network...\n")
history = model.fit(
    X_train_scaled, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

print("\n✓ Training complete!")

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history.history['accuracy'], label='Training Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy')
axes[0].set_title('Model Accuracy', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# Loss
axes[1].plot(history.history['loss'], label='Training Loss')
axes[1].plot(history.history['val_loss'], label='Validation Loss')
axes[1].set_title('Model Loss', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## Step 12: Evaluate Neural Network

In [ ]:
# Evaluate on test set
test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)

# Predictions
y_pred_nn_probs = model.predict(X_test_scaled)
y_pred_nn = np.argmax(y_pred_nn_probs, axis=1) if num_classes > 2 else (y_pred_nn_probs > 0.5).astype(int).flatten()

print("=" * 60)
print("NEURAL NETWORK RESULTS")
print("=" * 60)
print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_nn))

In [ ]:
# Confusion Matrix
cm_nn = confusion_matrix(y_test, y_pred_nn)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_nn, annot=True, fmt='d', cmap='Greens')
plt.title('Neural Network - Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

## Step 13: Model Comparison

In [ ]:
# Compare models
comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'Neural Network'],
    'Accuracy': [lr_accuracy, test_accuracy]
})

print("=" * 60)
print("MODEL COMPARISON")
print("=" * 60)
print(comparison.to_string(index=False))

# Visualization
plt.figure(figsize=(10, 6))
plt.bar(comparison['Model'], comparison['Accuracy'], color=['skyblue', 'lightgreen'])
plt.title('Model Accuracy Comparison', fontsize=14, fontweight='bold')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
for i, v in enumerate(comparison['Accuracy']):
    plt.text(i, v + 0.02, f'{v:.4f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

# Determine best model
best_model = comparison.loc[comparison['Accuracy'].idxmax(), 'Model']
best_accuracy = comparison['Accuracy'].max()
print(f"\n🏆 Best Model: {best_model} with {best_accuracy:.4f} accuracy")

## Step 14: Feature Importance Analysis

In [ ]:
# Feature importance from Logistic Regression
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': np.abs(lr_model.coef_[0]) if num_classes == 2 else np.abs(lr_model.coef_).mean(axis=0)
}).sort_values('Importance', ascending=False)

print("=" * 60)
print("TOP 10 MOST IMPORTANT FEATURES")
print("=" * 60)
print(feature_importance.head(10).to_string(index=False))

# Visualization
plt.figure(figsize=(10, 6))
top_10 = feature_importance.head(10)
plt.barh(top_10['Feature'], top_10['Importance'], color='coral')
plt.xlabel('Importance')
plt.title('Top 10 Feature Importance', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Step 15: Final Conclusions

### Summary of Findings:

Run all cells above, then write your conclusions here based on:
1. Which model performed better?
2. What are the top factors influencing student performance?
3. What insights can help improve student outcomes?
4. Any recommendations based on the analysis?

In [ ]:
# Final summary
print("=" * 60)
print("PROJECT SUMMARY")
print("=" * 60)
print(f"\nDataset: {df.shape[0]} students, {df.shape[1]} features")
print(f"\nBest Model: {best_model}")
print(f"Best Accuracy: {best_accuracy:.4f} ({best_accuracy*100:.2f}%)")
print(f"\nTop 3 Important Features:")
for i, row in feature_importance.head(3).iterrows():
    print(f"  {i+1}. {row['Feature']}")
print("\n✓ Analysis Complete!")